# Week 10 Lab — Machine Learning & Dashboards
**Name:** IREMBERE OLIVIER  
**Student ID:** 28392  
**course:** INTRO TO BIG DATA  
**GROUP:** A



## Part 1 · Explore & Split
### Exercise 1.1 — Know your data

In [2]:
import pandas as pd

df = pd.read_csv("week10_students.csv")
print("Shape:", df.shape)
print()
print(df[["quiz_avg", "assignments_avg", "attendance_pct", "score"]].corr()["score"].drop("score"))

# Most useful feature: quiz_avg (correlation ~0.80 with score), then assignments_avg (~0.54).
# attendance_pct is weakest (~0.24) on its own.

Shape: (150, 9)

quiz_avg           0.800726
assignments_avg    0.535983
attendance_pct     0.238118
Name: score, dtype: float64


### Exercise 1.2 — Frame & split

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("week10_students.csv")

# (a) features and target
X = df[["quiz_avg", "assignments_avg", "attendance_pct"]]
y = df["score"]

# (b) split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# (c) sizes
print("Train:", len(X_train), "| Test:", len(X_test))   # 112 / 38

# 25% of students are hidden from the model so we can test it on students it has never seen.
# Scoring a model on the same data it learned from would only measure memorisation;
# the hidden test set tells us how it will behave on real, new students.

Train: 112 | Test: 38


## Part 2 · Regression — Predict the Score
### Exercise 2.1 — Train & read the model

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

df = pd.read_csv("week10_students.csv")
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
y = df["score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

for name, coef in zip(features, model.coef_):
    print(f"{name:18s} {coef:8.4f}")
print(f"{'intercept':18s} {model.intercept_:8.4f}")

# One extra quiz point is worth about 0.58 exam points, holding assignments and attendance fixed.
# (Assignments: ~0.36 per point; attendance: ~0.11 per point.)

quiz_avg             0.5790
assignments_avg      0.3648
attendance_pct       0.1134
intercept           -4.4282


### Exercise 2.2 — Judge it honestly

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("week10_students.csv")
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
y = df["score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)          # TEST set only

print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R2 :", round(r2_score(y_test, pred), 2))

# MAE ~3.3: on students the model has never seen, its predicted score is off by about
#   3 points on average (e.g. it says 60, the real score is usually between 57 and 63).
# R2 ~0.89: the model explains about 89% of why students' scores differ from one another
#   using only quiz, assignment and attendance numbers; the other ~11% is things it cannot see.

MAE: 3.33
R2 : 0.89


### Exercise 2.3 — Predict a new student

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

df = pd.read_csv("week10_students.csv")
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
y = df["score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LinearRegression().fit(X_train, y_train)

new_student = pd.DataFrame([[70, 75, 90]], columns=features)   # quiz 70, assignments 75, attendance 90
print("New student predicted score:", round(model.predict(new_student)[0], 1))

# Me (invented numbers): quiz 60, assignments 65, attendance 85
me = pd.DataFrame([[60, 65, 85]], columns=features)
print("My predicted score:", round(model.predict(me)[0], 1))

New student predicted score: 73.7
My predicted score: 63.7


## Part 3 · Classification — Catch the At-Risk Students
### Exercise 3.1 — Build the target & train

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

df = pd.read_csv("week10_students.csv")

# (a) target
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())      # 1 = pass: 131, 0 = fail: 19

# (b) split + train
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
yc = df["passed"]
X_train, X_test, yc_train, yc_test = train_test_split(X, yc, test_size=0.25, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, yc_train)
print("Trained. Test size:", len(X_test))

passed
1    131
0     19
Name: count, dtype: int64
Trained. Test size: 38


### Exercise 3.2 — The accuracy trap

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

df = pd.read_csv("week10_students.csv")
df["passed"] = (df["score"] >= 50).astype(int)
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
yc = df["passed"]
X_train, X_test, yc_train, yc_test = train_test_split(X, yc, test_size=0.25, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(X_train, yc_train)
pred = clf.predict(X_test)

# (a) model accuracy
print("Model accuracy   :", round(accuracy_score(yc_test, pred), 3))
# (b) lazy baseline: predict "pass" for everyone
print("Lazy baseline    :", round(yc_test.mean(), 3))
# (c) confusion matrix (rows = actual [fail, pass], columns = predicted [fail, pass])
print(confusion_matrix(yc_test, pred))

# (d) No, the model is not useless. Both score 0.895 (34/38), but only because ~90% of students
# pass, so "everyone passes" looks great while being worthless: it catches 0 of the 4 failing
# students in the test set. The confusion matrix [[3, 1], [3, 31]] shows our model correctly flags
# 3 of those 4 at-risk students (75% recall on the class we care about), missing 1, at the cost
# of 3 false alarms (passing students flagged). Accuracy hides this; the matrix shows the model
# does the one job the baseline cannot: finding students who need help.

Model accuracy   : 0.895
Lazy baseline    : 0.895
[[ 3  1]
 [ 3 31]]


## Part 4 · Feature Engineering
### Exercise 4.1 — Does more information help?

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("week10_students.csv")
y = df["score"]
nums = ["quiz_avg", "assignments_avg", "attendance_pct"]

def test_r2(X):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    m = LinearRegression().fit(X_train, y_train)
    return r2_score(y_test, m.predict(X_test))

X1 = df[["quiz_avg"]]
X2 = df[nums]
X3 = pd.concat([df[nums], pd.get_dummies(df["program"])], axis=1)

results = pd.DataFrame({
    "version": ["1. quiz_avg only", "2. three numeric features", "3. numeric + program one-hot"],
    "test_R2": [test_r2(X1), test_r2(X2), test_r2(X3)],
}).round(3)
print(results)

# Helped: adding assignments_avg and attendance_pct (0.62 -> 0.89), a big jump, because they add
#   new, relevant information about how the student performs.
# Didn't help: adding program (0.89 -> 0.88). Which program a student is in tells the model almost
#   nothing beyond their actual marks, and extra columns can add noise.
# Lesson: more data is only better if it is new, relevant information. Test each addition on
#   unseen data instead of assuming that more features always means a better model.

                        version  test_R2
0              1. quiz_avg only    0.624
1     2. three numeric features    0.887
2  3. numeric + program one-hot    0.880


## Part 5 · Ship a Dashboard
### Exercise 5.1 — The story in three charts

In [9]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("week10_students.csv")

# (a)
fig1 = px.histogram(df, x="score", nbins=12, title="1. Where the class stands")
fig1.show()

In [10]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("week10_students.csv")
df["passed"] = (df["score"] >= 50).astype(int)

# (b) passed is 0/1, so make it categorical for a clean two-colour legend
fig2 = px.scatter(df, x="quiz_avg", y="score", color=df["passed"].astype(str), hover_name="name",
                  labels={"color": "passed"}, title="2. Who is at risk")
fig2.show()

In [11]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("week10_students.csv")

# (c)
avg = df.groupby("district", as_index=False)["score"].mean().sort_values("score")
fig3 = px.bar(avg, x="district", y="score", title="3. Where to act")
fig3.show()

### Exercise 5.2 — Publish it

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("week10_students.csv")
df["passed"] = (df["score"] >= 50).astype(int)

fig2 = px.scatter(df, x="quiz_avg", y="score", color=df["passed"].astype(str), hover_name="name",
                  labels={"color": "passed"}, title="2. Who is at risk")
fig2.write_html("class_dashboard.html")
print("Saved class_dashboard.html - open it in your browser, hover around, and screenshot it.")

Saved class_dashboard.html - open it in your browser, hover around, and screenshot it.


### Bonus (+3) — One chart, four stories

In [13]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("week10_students.csv")
fig = px.box(df, x="program", y="score", color="program", title="Score distribution by program")
fig.show()

# Networking is the most consistent (smallest spread, std ~10.8 and the narrowest middle half).
# Software Engineering has the widest spread (std ~13.8, scores from ~37 to ~93).

## Part 6 · The Model Card — Student Score Predictor (Linear Regression)

**1. Purpose.** Predicts a student's final score (0-100) for AUCA students, so lecturers can spot early who is likely to struggle and offer support sooner.

**2. Data & features.** 150 students (clean). Inputs: `quiz_avg`, `assignments_avg`, `attendance_pct`. Target: `score`. Trained on 112 students (75%) and tested on 38 unseen ones (25%), `random_state=42`.

**3. Performance (unseen students).** MAE ≈ 3.3 points: a prediction is typically about 3 points from the real score. R² ≈ 0.89: the model explains about 89% of the differences between students' scores. It is solid but not perfect, and it was tested on only 38 students, so these numbers are approximate.

**4. Risks.** A wrong prediction could label a capable student as failing (lowering expectations, stigma, missed opportunities) or miss a student who really needs help. It only sees three numbers, not illness, family problems or workload, and it was trained on one small group, so it may not fit other years or programs. *Use it* as an early-warning signal to start a conversation and offer support. *Do not use it* to decide grades, scholarships, discipline or admission, or without a human lecturer reviewing each case.
